# Imputation and segmentation evaluation

Evaluates the output of `imputation/run_imputation.py` (N imputed runs of one dropped modality per subject) and the
N x K segmentations of these runs (K prompts per run). Volumes are prepared with the
[BraSyn tutorial](https://github.com/WinstonHuTiger/BraSyn_tutorial) code, exactly as its `project/test.py` does
(IPL orientation, 144×192×192 crop, min-max to [0, 1]); segmentations are scored with the official
[BraTS evaluation](https://github.com/BraTS/BraTS_evaluation).

1. imputation: imputed runs vs real image (global and local SSIM, L1, L2, PSNR), spread of the N runs;
2. segmentation: BraTS metrics per (run, prompt) (Dice, NSD, lesion-wise F1), confidence map of the N x K masks;
3. ANOVA: Dice variability due to imputation vs prompt;
4. all subjects: local logistic regression giving a confidence heatmap, CSV of every metric.

Setup, once, from the repo root:
```
git clone https://github.com/WinstonHuTiger/BraSyn_tutorial external/BraSyn_tutorial
pip install BraTS-evaluation statsmodels
```
Run the notebook from the `trust_pipeline/` folder.

In [ ]:
import sys
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from skimage.metrics import peak_signal_noise_ratio as psnr  # what BraSyn's test.py imports
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

BRASYN_REPO = Path("../external/BraSyn_tutorial")
IMPUTED_DIR = Path("../results/W_preprocess_imputed_multiple_runs")  # output_dir in imputation/config.yaml
GT_DIR = Path("/home/dell/software/BraTs Africa/BraTS-Africa Dataset/BraTS-Africa_split/test")  # data_dir in imputation/config.yaml
MOD = "t1c"                                                           # the imputed modality to evaluate
PATCH = 3                                                             # local metrics: means over PATCH^3 voxels
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"               # full-size 3D SSIM on CPU can run out of RAM

sys.path.insert(0, str(BRASYN_REPO / "project"))
from util.ssim import create_window                                        # BraSyn's Gaussian SSIM window
from data.data_augmentation_3D import getBetterOrientation, toGrayScale    # BraSyn's preprocessing
from brats_evaluation import config_path, evaluate_single_exam             # BraTS_evaluation (pip install BraTS-evaluation)
from panoptica import Panoptica_Evaluator                                  # loads BraTS_evaluation's configs

EVALUATOR = Panoptica_Evaluator.load_from_config(str(config_path("gli")))   # BraTS glioma config

## Load one subject

`load` is BraSyn's dataloader transform (`data/brain_3D_random_mod_dataset.py`): reorient to IPL, then crop.
The real image is also rescaled to [0, 1] with `toGrayScale`; the imputed runs already are (they are BraSyn's output).

In [ ]:
def load(path):
    return getBetterOrientation(nib.load(path), "IPL").get_fdata()[8:152, 24:216, 24:216]


def load_subject(mod_dir):
    real = toGrayScale(load(next((mod_dir / "original").glob("*.nii*")))).astype(np.float32)
    imputed = np.stack([load(p) for p in sorted((mod_dir / "imputed").glob("*.nii*"))]).astype(np.float32)
    return real, imputed


mod_dirs = sorted(IMPUTED_DIR.glob(f"*/*/{MOD}"))   # <class>/<subject>/<mod>
mod_dir = mod_dirs[0]                                # change the index to pick another subject
real, imputed = load_subject(mod_dir)
print(mod_dir.parent.name, "| real", real.shape, "| imputed", imputed.shape, "(N runs, I, P, L)")

In [ ]:
z = real.shape[0] // 2   # middle axial slice (axis 0 is inferior-superior in IPL)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, img) in zip(axes, {"real": real, "imputed (run 0)": imputed[0], "std across runs": imputed.std(0)}.items()):
    ax.imshow(img[z], cmap="gray")
    ax.set_title(title)
    ax.axis("off")

## Metrics: global and local versions

Every voxel-wise metric map has two versions, inside the brain (`real > 0`):
- **global**: mean over the brain;
- **local**: mean over each `PATCH`³ patch (patch grid, NaN outside the brain), to localise the errors.

`ssim_map` is BraSyn's `util.ssim` (same Gaussian window and constants), which only returns the mean, kept per voxel.

In [ ]:
def ssim_map(a, b):
    x, y = (torch.from_numpy(np.ascontiguousarray(v, np.float32))[None, None].to(DEVICE) for v in (a, b))
    window = create_window(11, x.shape).to(x)

    def blur(v):
        return F.conv3d(v, window, padding=5)

    mx, my = blur(x), blur(y)
    vx, vy, cxy = blur(x * x) - mx ** 2, blur(y * y) - my ** 2, blur(x * y) - mx * my
    s = (2 * mx * my + 0.01 ** 2) * (2 * cxy + 0.03 ** 2) / ((mx ** 2 + my ** 2 + 0.01 ** 2) * (vx + vy + 0.03 ** 2))
    return s[0, 0].cpu().numpy()


def patches(volume, brain):
    """Local version: mean of each PATCH^3 patch over its brain voxels (NaN where it has none)."""
    n = [d // PATCH for d in volume.shape]

    def block_sum(v):
        return v.reshape(n[0], PATCH, n[1], PATCH, n[2], PATCH).sum((1, 3, 5))

    with np.errstate(invalid="ignore", divide="ignore"):
        return block_sum(volume * brain) / block_sum(brain.astype(np.float32))


def show(local_maps, z, cmap="viridis"):
    """Axial slice z (in voxels) of each local (patch) map."""
    fig, axes = plt.subplots(1, len(local_maps), figsize=(4 * len(local_maps), 4))
    for ax, (title, m) in zip(np.atleast_1d(axes), local_maps.items()):
        im = ax.imshow(m[z // PATCH], cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046)

## 1a. Imputed vs real

Each run against the real image: SSIM, L1 and L2 (global and local, the local maps averaged over the runs), and PSNR
as in BraSyn's `test.py` (`data_range` = range of the imputed image).

In [ ]:
def compare_to_real(real, imputed):
    brain, rows, local = real > 0, [], {}
    for run in imputed:
        maps = {"SSIM": ssim_map(real, run), "L1": np.abs(run - real), "L2": (run - real) ** 2}
        rows.append({**{k: float(m[brain].mean()) for k, m in maps.items()},
                     "PSNR": psnr(real, run, data_range=run.max() - run.min())})
        for k, m in maps.items():
            local[k] = local.get(k, 0) + patches(m, brain) / len(imputed)
    return pd.DataFrame(rows), local


per_run, local_quality = compare_to_real(real, imputed)
show({f"local {k}": m for k, m in local_quality.items()}, z)
per_run.agg(["mean", "std"]).T

## 1b. Spread across the N runs (no ground truth needed)

Same input, N outputs: voxel-wise variance across the runs, SSIM and PSNR between every pair of runs (global and local;
the local PSNR of a pair uses the mean squared difference of each patch).

In [ ]:
def spread(real, imputed):
    brain, pairs = real > 0, list(combinations(imputed, 2))
    variance, pair_ssim, pair_psnr, ssim_values = imputed.var(0), 0, 0, []
    for a, b in pairs:
        m = ssim_map(a, b)
        ssim_values.append(float(m[brain].mean()))
        pair_ssim = pair_ssim + m / len(pairs)
        mse = np.maximum(patches((a - b) ** 2, brain), 1e-10)                  # identical patch: 100 dB, not inf
        pair_psnr = pair_psnr + 10 * np.log10((b.max() - b.min()) ** 2 / mse) / len(pairs)
    summary = {"mean variance": float(variance[brain].mean()), "pairwise SSIM": np.mean(ssim_values),
               "pairwise PSNR": np.mean([psnr(a, b, data_range=b.max() - b.min()) for a, b in pairs])}
    return summary, {"variance": patches(variance, brain), "pairwise SSIM": patches(pair_ssim, brain),
                     "pairwise PSNR": pair_psnr}


summary, local_spread = spread(real, imputed)
show({f"local {k}": m for k, m in local_spread.items()}, z)
pd.Series(summary)

## 2. Segmentation

One mask per (imputed run n, prompt k), next to the imputed runs:
`<class>/<subject>/<mod>/segmentation/<subject>-<mod>-runNN-promptKK.nii.gz` (whole tumour).
Each mask is scored against the reference `<subject>-seg.nii*` with `brats_evaluation.evaluate_single_exam`
(config `gli`, whole-tumour group). Dice and NSD measure the whole mask; the lesion-wise F1 measures lesion
detection (lesions found, missed or invented): comparing them tells whether the errors are global (boundary)
or local (missed or invented lesions).

In [ ]:
def load_segmentations(mod_dir, n_runs):
    gt = load(next(GT_DIR.rglob(f"{mod_dir.parent.name}-seg.nii*"))).astype(np.uint8)   # BraTS label map
    files = sorted((mod_dir / "segmentation").glob("*.nii*"))                          # runNN-promptKK, run-major
    return gt, np.stack([load(f) > 0.5 for f in files]).reshape(n_runs, -1, *gt.shape)


def segment_scores(gt, masks):
    rows = []
    for n, k in np.ndindex(masks.shape[:2]):
        wt = evaluate_single_exam(masks[n, k].astype(np.uint8), gt, f"run{n}-prompt{k}", EVALUATOR)["wt"]
        rows.append({"n": n, "k": k, "dice": wt["global_bin_dsc"], "nsd": wt["global_bin_nsd"],
                     "lesion-wise F1": wt["rq"]})
    return pd.DataFrame(rows)


gt, masks = load_segmentations(mod_dir, len(imputed))
per_mask = segment_scores(gt, masks)
per_mask.drop(columns=["n", "k"]).agg(["mean", "std"]).T

### Confidence map of the N x K segmentations

Fraction of the N x K masks that call each voxel tumour: 0 or 1 where they all agree, in between where the result
depends on the imputed run or the prompt (mostly at the edges). Red: reference contour.

In [ ]:
def agreement(masks):
    return masks.mean((0, 1))


p = agreement(masks)
zt = int(np.argmax((gt > 0).sum((1, 2))))   # axial slice with the largest tumour
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(real[zt], cmap="gray")
im = ax.imshow(np.ma.masked_equal(p[zt], 0), cmap="viridis", vmin=0, vmax=1, alpha=0.8)
ax.contour(gt[zt] > 0, levels=[0.5], colors="red", linewidths=1)
ax.set_title("agreement of the N x K masks")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046)

## 3. Imputation × segmentation

Two-way ANOVA of the Dice per (n, k), to split the variability between imputation and prompt
(one mask per cell, so the residual is the interaction).

In [ ]:
anova = anova_lm(ols("dice ~ C(n) + C(k)", per_mask).fit())
anova.assign(share=anova["sum_sq"] / anova["sum_sq"].sum())

## 4. All subjects: metrics and local confidence model

Per patch of the brain, local features computed **without** the ground truth: SSIM, PSNR and variance between the
runs (imputation); pairwise Dice and NSD between the MedSAM passes (segmentation). Label (needs the ground truth):
the consensus segmentation (p ≥ 0.5) is right on at least 90 % of the patch. A logistic regression trained on the
other subjects gives each patch of a subject a confidence score in [0, 1]: P(segmentation right).

In [ ]:
def local_pass_agreement(masks, brain):
    """Pairwise Dice and NSD (tolerance 1 voxel) between the K passes of each run, per patch, averaged over the
    pairs and the runs (1 where the patch has no mask voxel / no boundary)."""
    dice, nsd, n_pairs = 0, 0, 0
    for masks_n in masks:
        surface = [m & ~ndi.binary_erosion(m) for m in masks_n]
        near = [ndi.distance_transform_edt(~s) <= 1 if s.any() else np.zeros_like(s) for s in surface]
        for i, j in combinations(range(len(masks_n)), 2):
            a, b = masks_n[i], masks_n[j]
            with np.errstate(invalid="ignore", divide="ignore"):
                d = 2 * patches(a & b, brain) / (patches(a, brain) + patches(b, brain))
                s = ((patches(surface[i] & near[j], brain) + patches(surface[j] & near[i], brain))
                     / (patches(surface[i], brain) + patches(surface[j], brain)))
            dice, nsd, n_pairs = dice + np.nan_to_num(d, nan=1.0), nsd + np.nan_to_num(s, nan=1.0), n_pairs + 1
    return dice / n_pairs, nsd / n_pairs


def patch_table(real, gt, masks, local_spread):
    brain, p = real > 0, agreement(masks)
    roi = patches(brain.astype(np.float32), brain) > 0                             # every patch with brain voxels
    pair_dice, pair_nsd = local_pass_agreement(masks, brain)
    features = {"SSIM": local_spread["pairwise SSIM"], "PSNR": local_spread["pairwise PSNR"],
                "variance": local_spread["variance"], "pairwise Dice": pair_dice, "pairwise NSD": pair_nsd}
    correct = patches(((p >= 0.5) == (gt > 0)).astype(np.float32), brain) >= 0.9
    return pd.DataFrame({**{k: f[roi] for k, f in features.items()}, "correct": correct[roi]}), roi


rows, tables = [], {}
for d in mod_dirs:
    real, imputed = load_subject(d)
    gt, masks = load_segmentations(d, len(imputed))
    per_run, _ = compare_to_real(real, imputed)
    summary, local_spread = spread(real, imputed)
    per_mask = segment_scores(gt, masks)
    anova = anova_lm(ols("dice ~ C(n) + C(k)", per_mask).fit())
    tables[d.parent.name] = patch_table(real, gt, masks, local_spread)
    rows.append({"subject": d.parent.name, "n_runs": len(imputed), **per_run.mean().add_suffix(" (mean over runs)"),
                 **summary, **per_mask.drop(columns=["n", "k"]).mean().add_suffix(" (mean over masks)"),
                 **(anova["sum_sq"] / anova["sum_sq"].sum()).rename(lambda s: f"Dice variance share {s}")})
results = pd.DataFrame(rows).set_index("subject")

### Local logistic regression → confidence heatmap

Leave one subject out: the model is fitted on the other subjects' patches, then scores the left-out subject.
AUROC: how well 1 − confidence finds the patches where the segmentation is wrong.

In [ ]:
FEATURES = ["SSIM", "PSNR", "variance", "pairwise Dice", "pairwise NSD"]
confidence = {}
for s, (table, roi) in tables.items():
    train = pd.concat([t for o, (t, _) in tables.items() if o != s])
    model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced"))
    model.fit(train[FEATURES], train["correct"])
    score = model.predict_proba(table[FEATURES])[:, 1]
    results.loc[s, "confidence AUROC"] = (roc_auc_score(~table["correct"], 1 - score)
                                          if table["correct"].nunique() == 2 else np.nan)
    confidence[s] = np.full(roi.shape, np.nan)
    confidence[s][roi] = score

s = mod_dir.parent.name                 # the subject of sections 1-3, at its largest tumour slice zt
plt.imshow(confidence[s][zt // PATCH], cmap="RdYlBu", vmin=0, vmax=1)
plt.title(f"{s}: confidence P(segmentation right) per patch")
plt.axis("off")
plt.colorbar(fraction=0.046)
results.to_csv(f"metrics_{MOD}.csv")
results